# Step 14.1 — Residual Connection：为什么 Transformer 不只计算 `F(x)`，而是 `x + F(x)`？

Step 13 结束时，Multi-Head Attention 已经形成一个 shape-preserving transformation：

$$
x:(B,T,C)
\quad\longrightarrow\quad
F(x):(B,T,C).
$$

这里先把整个 Attention module 简写为：

$$
F(x).
$$

Transformer block 并不会简单地把旧 representation `x` 丢掉，只保留 `F(x)`。

而是使用：

$$
\boxed{y=x+F(x)}.
$$

这叫 **residual connection / skip connection（残差连接 / 跳跃连接）**。

本节只解释 residual：shape requirement、update-as-correction、gradient highway 直觉，以及它的数学边界；不进入 Step 14.2 的 LayerNorm / RMSNorm。


## 1. 为什么 Step 13 的 `(B,T,C) -> (B,T,C)` 正好适合 Residual？

Element-wise addition 要求 `x` 与 `F(x)` 的对应元素能够一一相加。

当前：

$$
x:(B,T,C),
$$

而 Multi-Head Attention 最终经过 `c_proj` 后：

$$
F(x):(B,T,C).
$$

所以可以直接：

$$
y_{b,t,c}=x_{b,t,c}+F(x)_{b,t,c}.
$$

Shape 保持：

$$
\boxed{(B,T,C)+(B,T,C)\rightarrow(B,T,C)}.
$$

这也是为什么 Transformer 的许多 sublayers 会设计成输入/输出保持 model width `C`。

### 不要把 Broadcasting 当成这里的核心

某些 array library 的 `+` 支持 broadcasting，但标准 Transformer residual connection 的概念是**同 shape representation 的逐元素相加**。

本节不依赖广播来“凑”shape。


## 2. Update-as-Correction：让 `F(x)` 学“改多少”，而不是每次重建全部表示

把：

$$
y=x+F(x)
$$

换一个角度看。

`x` 可以理解为当前 representation baseline，而：

$$
\Delta x=F(x)
$$

是这个 sublayer 建议施加的 **correction / update（修正量 / 更新量）**。

于是：

$$
\boxed{y=x+\Delta x}.
$$

如果当前 representation 已经大致合理，sublayer 可以学习一个较小 correction；如果需要大幅改变，也可以输出较大的 correction。

这种说法是有用的优化直觉，但不是说实际训练一定强制 `F(x)` 很小，也不是说每层都只能做微小变化。


## 3. Tiny Vector Example：旧表示 + 修正量

假设某个 token 当前 representation 只有两个 features：

$$
x=[2,-1].
$$

Attention sublayer 给出的 correction：

$$
F(x)=[-0.5,0.25].
$$

那么 residual output：

$$
y=x+F(x)=[2,-1]+[-0.5,0.25]=\boxed{[1.5,-0.75]}.
$$

这里可以直接看到：

```text
old state      [ 2.00, -1.00]
correction     [-0.50, +0.25]
--------------------------------
new state      [ 1.50, -0.75]
```

Residual connection 并不是“复制 x 后什么都不做”；真正输出仍然由 learned `F(x)` 改写。


In [ ]:
import mlx.core as mx

x = mx.array([2.0, -1.0])
fx = mx.array([-0.5, 0.25])

y = x + fx

print('x   :', x)
print('F(x):', fx)
print('y   :', y)
print('shape:', y.shape)

# Expected mathematically: y = [1.5, -0.75]
# This master notebook remains unexecuted.


## 4. Gradient Highway：为什么 Identity Path 对 Backprop 有帮助？

现在看：

$$
y=x+F(x).
$$

对 x 求 Jacobian：

$$
\frac{\partial y}{\partial x}=\frac{\partial x}{\partial x}+\frac{\partial F(x)}{\partial x}.
$$

而：

$$
\frac{\partial x}{\partial x}=I.
$$

所以：

$$
\boxed{J_y=I+J_F}.
$$

如果上游 gradient 是 column-vector convention 的：

$$
g_y=\frac{\partial L}{\partial y},
$$

那么 backprop：

$$
g_x=(I+J_F)^Tg_y=\boxed{g_y+J_F^Tg_y}.
$$

第一项 `g_y` 来自 residual 的 **identity path（恒等路径）**；第二项 `J_F^Tg_y` 来自 learned transformation F。

这就是 **gradient highway（梯度高速通道）** 的核心直觉：gradient 不必只经过复杂的 F 分支，还多了一条直接 identity contribution。


## 5. 一个两维 Gradient Example

假设在某个点：

$$
J_F=\begin{bmatrix}-0.5&0\\0&0.2\end{bmatrix},
\qquad
g_y=\begin{bmatrix}1\\2\end{bmatrix}.
$$

Residual identity path 直接贡献：

$$
g_y=\begin{bmatrix}1\\2\end{bmatrix}.
$$

F branch 贡献：

$$
J_F^Tg_y=\begin{bmatrix}-0.5\\0.4\end{bmatrix}.
$$

所以：

$$
g_x=\begin{bmatrix}1\\2\end{bmatrix}+\begin{bmatrix}-0.5\\0.4\end{bmatrix}=\boxed{\begin{bmatrix}0.5\\2.4\end{bmatrix}}.
$$

重点不是这个具体数字，而是：backprop 中存在一项不经过 `J_F` 的直接 `g_y` contribution。


In [ ]:
J_F = mx.array([
    [-0.5, 0.0],
    [ 0.0, 0.2],
])
g_y = mx.array([1.0, 2.0])

identity_contribution = g_y
branch_contribution = mx.swapaxes(J_F, -1, -2) @ g_y
g_x = identity_contribution + branch_contribution

print('identity contribution:', identity_contribution)
print('F-branch contribution :', branch_contribution)
print('total g_x             :', g_x)

# Expected mathematically:
# identity contribution = [1.0, 2.0]
# branch contribution   = [-0.5, 0.4]
# total                 = [0.5, 2.4]
# Master notebook remains unexecuted.


## 6. 重要边界：Gradient Highway 不是“梯度永远不会消失”的保证

公式：

$$
g_x=g_y+J_F^Tg_y
$$

确实存在 direct identity term。

但第二项可能与第一项同方向加强、不同方向改变、部分抵消，甚至极端情况下完全抵消。

最简单的反例：

$$
F(x)=-x.
$$

于是：

$$
y=x+(-x)=0.
$$

而：

$$
J_F=-I.
$$

所以：

$$
I+J_F=I-I=0.
$$

反向：

$$
g_x=0.
$$

所以 residual connection 提供有利的 identity path，但数学上**不保证任何参数状态下 gradient 必然非零**。

在很多层堆叠时，各层的 `(I+J_F)` 仍然会连乘；整体梯度行为依然取决于 learned Jacobians、normalization、optimization 等因素。


## 7. Residual 也不保证“信息零损失”或可逆

看到 `y=x+F(x)`，很容易误以为：

> “既然 x 被加回去了，那原始信息一定完整保存。”

这也不是数学保证。

刚才反例：

$$
F(x)=-x
$$

直接得到：

$$
y=0.
$$

不同输入都可能映射到同一个输出，显然丢失信息。

所以 residual 的正确表述是：

> **它提供一条显式 identity route，使 sublayer 可以在已有 representation 上学习 correction，并为前向信息流和反向梯度流提供更直接的路径。**

但不能由此推出 mapping 必然可逆、representation 必然无损、gradient 必然稳定，或网络再深也不会优化困难。


## 8. 回到 Transformer：Attention 输出为什么特别适合 Residual？

Step 13 的 Attention module：

$$
\operatorname{MHA}(x):(B,T,C)
$$

和输入：

$$
x:(B,T,C)
$$

shape 完全一致。

因此最基本的 residual skeleton 可以写成：

```python
attention_update = attention(x)
x = x + attention_update
```

概念上：

```text
x
│
├──────────── identity path ────────────┐
│                                       │
└─ Attention / F(x) -> correction ──────┤
                                        ↓
                                     x + F(x)
```

这里暂时只看 residual topology。

真实现代 Transformer 通常还会把 normalization 放在特定位置，但那正是 Step 14.2–14.3 的内容，本轮不提前展开。


In [ ]:
# Shape-only residual example matching Transformer notation.
B, T, C = 2, 4, 6

x_seq = mx.zeros((B, T, C))
attention_update = mx.zeros((B, T, C))

y_seq = x_seq + attention_update

print('x:', x_seq.shape)
print('F(x):', attention_update.shape)
print('x + F(x):', y_seq.shape)

# Residual addition itself introduces no trainable parameters.
# No normalization or Transformer Block is introduced in Step 14.1.


## 9. Residual Addition 本身有 Parameters 吗？没有

操作：

$$
y=x+F(x)
$$

中的 `+` 本身没有 learned weights。

Trainable parameters 属于 F，例如前一大 Step 的 Attention projections。

Residual path 本身只是：

```text
identity(x) = x
```

因此它不会因为“加了一条 skip connection”就额外增加一个参数矩阵。

后续如果某个 architecture 在 shape 不同情况下使用 learned projection shortcut，那是另一种设计；当前 Transformer residual 的核心情况是输入输出同为 `(B,T,C)`，可以直接逐元素相加。


## 10. Step 14.1 材料边界

- Transformer residual connection 的核心形式是 `y = x + F(x)`，本节 F(x) 可理解为已经完成的 Multi-Head Attention output。
- 当前 x 与 F(x) 都是 `(B,T,C)`，因此 residual 是同 shape element-wise addition；不依赖 broadcasting 来匹配维度。
- update-as-correction 是优化/表示直觉：F(x) 提供对当前 representation 的 learned update，但不要求 correction 必然很小。
- Jacobian 为 `I + J_F`；反向为 `g_x = g_y + J_F^T g_y`，其中 `g_y` 是 identity path 的 direct gradient contribution。
- gradient highway 是额外直接路径，不保证整个网络绝不会 gradient vanishing / exploding；`F(x)=-x` 是直接反例。
- residual 也不保证零信息损失或可逆；`F(x)=-x` 时所有输入都得到 `y=0`。
- residual addition 自身不增加 learned parameters。
- 本轮只准备 Step 14.1；不进入 Step 14.2，不讲 LayerNorm、RMSNorm、RMS/epsilon/gamma，也不展开 Pre-Norm / Post-Norm。
